In [ ]:
"""
====================================================================================
Ethereum Block Receipts Extraction via Infura
====================================================================================

Purpose
-------
This module retrieves transaction receipts for specific Ethereum blocks using the
Infura JSON-RPC API. The receipts provide detailed information about all transactions
included in each block, including logs, status, gas used, and contract events.

Workflow
--------
1. Identify all distinct block numbers from a preprocessed dataset.
2. Convert block numbers to hexadecimal format required by Ethereum JSON-RPC.
3. Query the Infura API using the `eth_getBlockReceipts` method for each block.
4. Aggregate all retrieved receipts into a single DataFrame.

Inputs
------
- `processed_blocknative_data`: a DataFrame containing a column `curblocknumber`
  with the list of blocks to fetch receipts for.
- Infura API key (string) to access the Ethereum mainnet endpoint.

Outputs
-------
- `receipts_df`: a consolidated pandas DataFrame containing the receipts for all
  specified blocks. Can be empty if no data is returned.

Rate Limiting
-------------
- Requests include a 2-second delay between consecutive calls to respect Infura's
  rate limits.

Use Case
--------
- This dataset can be used for transaction-level analysis, MEV research, smart
  contract monitoring, and DeFi activity analysis.
"""

In [ ]:
import pandas as pd
import requests
import json
import time

unique_blocks = processed_blocknative_data['curblocknumber'].unique()
block_hexes = [hex(b) for b in unique_blocks]
df_block_hexes = pd.DataFrame(block_hexes, columns=['block_hex'])

API_KEY = "API_KEY_HERE"
url = f"https://mainnet.infura.io/v3/{API_KEY}"
headers = {"Content-Type": "application/json"}

all_receipts = []
block_hex_list = df_block_hexes.iloc[:, 0].tolist()

for i, block_hex in enumerate(block_hex_list, start=1):
    payload = {
        "jsonrpc": "2.0",
        "method": "eth_getBlockReceipts",
        "params": [block_hex],
        "id": i
    }
    
    try:
        response = requests.post(url, headers=headers, data=json.dumps(payload))
        response.raise_for_status()
        data = response.json()
        
        if 'result' in data and data['result'] is not None:
            all_receipts.append(pd.DataFrame(data['result']))
    except Exception:
        pass

    time.sleep(2)

if all_receipts:
    receipts_df = pd.concat(all_receipts, ignore_index=True)
else:
    receipts_df = pd.DataFrame()
